In [2]:
import chromadb
from chromadb.config import Settings
from typing import List, Dict, Any

from openai import api_key
from sentence_transformers import SentenceTransformer


class ChromaVectorDBService:
    def __init__(self, persist_dir: str = './chroma-db' , model_name: str = 'all-MiniLM-L6-v2',):
        self.client = chromadb.PersistentClient(path="chroma_db")
        # self.collection = self.chroma_client.get_or_create_collection(name=collection_name)
        self.model = SentenceTransformer(model_name)

    ## Get Embeddings
    def get_embedding(self,text:str) -> List[float]:
        return self.model.encode(text).tolist()

    ## get_collections
    def get_collection(self,collection_name: str):
        return self.client.get_or_create_collection(name=collection_name)

    ## Load Data into vector store
    def load_data(self, doc_id, meta_data, collection_name: str, text: str):
        collection = self.get_collection(collection_name)
        embeddings = self.get_embedding(text)
        collection.add(documents=[text], embeddings=[embeddings], metadatas=[meta_data], ids=[doc_id])

        return {
            "id": doc_id,
            "text": text,
            "meta_data": meta_data,
            "embeddings": embeddings,
            "collection_name": collection_name,
            "status":"success"
        }

    ## clear collection
    def clear_collection(self,collection_name: str):
        collection = self.get_collection(collection_name)
        existing_ids=collection.get()["ids"]
        if existing_ids:
            collection.delete(ids=existing_ids)

    ## add resume
    def add_resume(self, resume_id: str, text: str, metadata):
        return self.load_data(collection_name="resume", doc_id=resume_id, meta_data=metadata, text=text)

    ## add JobPost
    def add_job_post(self, job_post_id: str, text: str, metadata):
        return self.load_data(collection_name="job_post", doc_id=job_post_id, meta_data=metadata, text=text)

    ## search jobs using resume embeddings
    def search_jobs_for_resume(self,resume_text: str, k=2):
        embeddings=self.get_embedding(resume_text)
        jobs = self.get_collection("job_post").query(
            query_embeddings=[embeddings],
            n_results=k
        )

        output=[]

        for doc, meta, dist, id_  in zip(jobs['documents'], jobs['metadatas'], jobs['distances'], jobs['ids']):
            output.append({
                "id": id_,
                "text": doc,
                "meta_data": meta,
                "distance": dist,
                "collection_name": "job_post",
            })
        return output

    ## search resumes for jobs
    def search_resumes_for_job(self,job_text: str, k=5):
        embeddings=self.get_embedding(job_text)
        resumes = self.get_collection("resume").query(
            query_embeddings=[embeddings],
            n_results=k
        )


        output=[]


        for doc,meta,dist, id_ in zip(resumes.get('documents'), resumes.get('metadatas'), resumes.get('distances'), resumes.get('ids')):

            output.append({
                "id": id_,
                "text": doc,
                "meta_data": meta,
                "distance": dist,
                "collection_name": "resume"
            })
        return output



    ### update the docs
    def update_docs(self,collection_name: str, doc_id: str, text: str, metadata):
        collection = self.get_collection(collection_name)
        collection.delete(ids=[doc_id])
        embeddings = self.get_embedding(text)
        collection.add(documents=[text], embeddings=[embeddings], metadatas=[metadata], ids=[doc_id])

        return {
            "id": doc_id,
            "text": text,
            "meta_data": metadata,
            "embeddings": embeddings,
            "collection_name": collection_name,
            "status":"success"
        }


    ## refresh job_post


chroma_db_service = ChromaVectorDBService()

In [3]:
data={'location': 'Noida, U.P India', 'skills': ['Analytical Skills', 'Quick Learner', 'C++', 'Statistics', 'SQL', 'TensorFlow', 'Machine Learning', 'Team Collaboration', 'Data Structures', 'SDLC', 'Python', 'Computer Vision', 'MATLAB', 'Deep Learning', 'Neural Network', 'NLP'], 'total_experience': '0 years', 'work_experience': [{'title': 'Research Intern', 'company': 'IIT Patna', 'position': 'Research Intern', 'start_date': 'Feb 2023', 'end_date': 'April 2023', 'description': 'Worked on Vision transformer, deep learning, Neural Network'}, {'title': 'Machine learning Intern', 'company': 'Ignitus', 'position': 'Machine learning Intern', 'start_date': 'June 2023', 'end_date': 'June 2023', 'description': 'Worked on PCA Algorithm with EDA on NMIST dataset.'}], 'education': [{'degree': 'Bachelor of Technology', 'institute': 'AMITY UNIVERSITY', 'field_of_study': 'Computer Science', 'start_date': '2020', 'end_date': '2024'}], 'certifications': ['Introduction to Generative AI Learning Path - Google_Cloud_Badge', 'UHACKATHON 3.0 -UPES Dehradun', 'AWS AI ML Deep racer Student league –AWS-deepracer', 'AI Programming with Python - Bertelsmann Nanodegree Scholarship on Udacity'], 'projects': [{'title': 'Liver segmentation using monai to detect tumor (In-house Internship College)', 'description': 'Worked on healthcare imaging with use of AI (Monai and Pytorch).\nUsed UNet, Image Segmentation, object detection, linear regression, loss function etc in project.\nGot to detect tumor through given CT/MRI scans.'}, {'title': 'Deep Convolutional Generative Adversarial Network (DCGAN) for Realistic Human Face Generation', 'description': 'Developed a DCGAN model for generating lifelike human faces.\nUtilized a dataset of real face images and pre-processed them for consistency.\nDesigned and implemented the generator and discriminator architecture.\nTrained the DCGAN iteratively to produce high-quality synthetic images, demonstrating proficiency in GANs and deep learning techniques .'}, {'title': 'Toxic Comment Classifier using NLP and ML', 'description': 'This Project aims Towards Classifying English Comments in 6 Different Categories\n(Toxic, Severe Toxic, Threat, Obscene, Insult and Identity Hate) Including their Neutral\nCases using Concepts of NLP and ML'}, {'title': 'Morse Code Generator in MATLAB', 'description': 'This Project Aims Towards Generating Morse Code of Letters, Words , Sentences with Audio.\nTTS (Using .NET Framework) is Included which will Translate those Morse codes as dot, dashes, letter, and word.'}], 'interests': []}

In [4]:
import json
data=json.loads(data)

TypeError: the JSON object must be str, bytes or bytearray, not dict

In [5]:
type(data)

dict

In [6]:
data

{'location': 'Noida, U.P India',
 'skills': ['Analytical Skills',
  'Quick Learner',
  'C++',
  'Statistics',
  'SQL',
  'TensorFlow',
  'Machine Learning',
  'Team Collaboration',
  'Data Structures',
  'SDLC',
  'Python',
  'Computer Vision',
  'MATLAB',
  'Deep Learning',
  'Neural Network',
  'NLP'],
 'total_experience': '0 years',
 'work_experience': [{'title': 'Research Intern',
   'company': 'IIT Patna',
   'position': 'Research Intern',
   'start_date': 'Feb 2023',
   'end_date': 'April 2023',
   'description': 'Worked on Vision transformer, deep learning, Neural Network'},
  {'title': 'Machine learning Intern',
   'company': 'Ignitus',
   'position': 'Machine learning Intern',
   'start_date': 'June 2023',
   'end_date': 'June 2023',
   'description': 'Worked on PCA Algorithm with EDA on NMIST dataset.'}],
 'education': [{'degree': 'Bachelor of Technology',
   'institute': 'AMITY UNIVERSITY',
   'field_of_study': 'Computer Science',
   'start_date': '2020',
   'end_date': '202

In [21]:
def format_resume_text(resume: Dict[str, Any]) -> str:
    lines = []

    lines.append(f"Location: {resume.get('location', '')}")
    lines.append(f"Total Experience: {resume.get('total_experience', '')}")
    lines.append("Skills: " + ", ".join(resume.get("skills", [])))

    lines.append("\nWork Experience:")
    for exp in resume.get("work_experience", []):
        lines.append(f"- {exp['title']} at {exp['company']} ({exp['start_date']} to {exp['end_date']}): {exp['description']}")

    lines.append("\nEducation:")
    for edu in resume.get("education", []):
        lines.append(f"- {edu['degree']} in {edu['field_of_study']} from {edu['institute']} ({edu['start_date']}–{edu['end_date']})")

    lines.append("\nCertifications:")
    for cert in resume.get("certifications", []):
        lines.append(f"- {cert}")

    lines.append("\nProjects:")
    for proj in resume.get("projects", []):
        lines.append(f"- {proj['title']}: {proj['description']}")

    return "\n".join(lines)

In [22]:
resume_id = "resume_salil_001"
resume_text = format_resume_text(data)
metadata = {
    "ids":resume_id
}
# chroma_db_service.add_resume(resume_id=resume_id, text=resume_text, metadata=metadata)

In [23]:
chroma_db_service.load_data(
    doc_id=resume_id, text=resume_text, meta_data=metadata, collection_name="resume"+resume_id
)

{'id': 'resume_salil_001',
 'text': 'Location: Noida, U.P India\nTotal Experience: 0 years\nSkills: Analytical Skills, Quick Learner, C++, Statistics, SQL, TensorFlow, Machine Learning, Team Collaboration, Data Structures, SDLC, Python, Computer Vision, MATLAB, Deep Learning, Neural Network, NLP\n\nWork Experience:\n- Research Intern at IIT Patna (Feb 2023 to April 2023): Worked on Vision transformer, deep learning, Neural Network\n- Machine learning Intern at Ignitus (June 2023 to June 2023): Worked on PCA Algorithm with EDA on NMIST dataset.\n\nEducation:\n- Bachelor of Technology in Computer Science from AMITY UNIVERSITY (2020–2024)\n\nCertifications:\n- Introduction to Generative AI Learning Path - Google_Cloud_Badge\n- UHACKATHON 3.0 -UPES Dehradun\n- AWS AI ML Deep racer Student league –AWS-deepracer\n- AI Programming with Python - Bertelsmann Nanodegree Scholarship on Udacity\n\nProjects:\n- Liver segmentation using monai to detect tumor (In-house Internship College): Worked on 

In [24]:
chroma_db_service.get_collection()

TypeError: ChromaVectorDBService.get_collection() missing 1 required positional argument: 'collection_name'

In [25]:
collection = chroma_db_service.get_collection("resume")
print(collection.get()['documents'])  # Should show empty 'documents' and 'ids'


['Location: Noida, U.P India\nTotal Experience: 0 years\nSkills: Analytical Skills, Quick Learner, C++, Statistics, SQL, TensorFlow, Machine Learning, Team Collaboration, Data Structures, SDLC, Python, Computer Vision, MATLAB, Deep Learning, Neural Network, NLP\n\nWork Experience:\n- Research Intern at IIT Patna (Feb 2023 to April 2023): Worked on Vision transformer, deep learning, Neural Network\n- Machine learning Intern at Ignitus (June 2023 to June 2023): Worked on PCA Algorithm with EDA on NMIST dataset.\n\nEducation:\n- Bachelor of Technology in Computer Science from AMITY UNIVERSITY (2020–2024)\n\nCertifications:\n- Introduction to Generative AI Learning Path - Google_Cloud_Badge\n- UHACKATHON 3.0 -UPES Dehradun\n- AWS AI ML Deep racer Student league –AWS-deepracer\n- AI Programming with Python - Bertelsmann Nanodegree Scholarship on Udacity\n\nProjects:\n- Liver segmentation using monai to detect tumor (In-house Internship College): Worked on healthcare imaging with use of AI (

In [26]:
job_posts = [
    {
        "id": "job_001",
        "text": "We are hiring a backend developer with experience in Django, PostgreSQL, and REST APIs.",
        "metadata": {
            "role": "Backend Developer",
            "location": "Bangalore",
            "company": "CodeCraft",
            "experience_required": "1+ years"
        }
    },
    {
        "id": "job_002",
        "text": "Looking for a data analyst proficient in SQL, Excel, and data visualization tools like Tableau.",
        "metadata": {
            "role": "Data Analyst",
            "location": "Noida",
            "company": "InsightEdge",
            "experience_required": "0-2 years"
        }
    },
    {
        "id": "job_003",
        "text": "Join our AI research team as an NLP Engineer. Must have experience with Transformers and PyTorch.",
        "metadata": {
            "role": "NLP Engineer",
            "location": "Remote",
            "company": "DeepLang AI",
            "experience_required": "2+ years"
        }
    }
]

for job_post in job_posts:
    chroma_db_service.add_job_post(
        job_post_id=job_post["id"],
        text=job_post["text"],
        metadata=job_post["metadata"],
    )


In [27]:
# chroma_db_service.get_collection('job_posts')
collection = chroma_db_service.get_collection('job_post')

In [28]:
match=chroma_db_service.search_jobs_for_resume(resume_text)

In [29]:
match_1=chroma_db_service.search_resumes_for_job("We are hiring a backend developer with experience in Django, PostgreSQL, and REST APIs")

In [30]:
m = list(match_1[0]['id']+match_1[0][t] for t in match_1[0] if t == 'text')
for i in m:
    print(i[1])

Location: Noida, U.P India
Total Experience: 0 years
Skills: Analytical Skills, Quick Learner, C++, Statistics, SQL, TensorFlow, Machine Learning, Team Collaboration, Data Structures, SDLC, Python, Computer Vision, MATLAB, Deep Learning, Neural Network, NLP

Work Experience:
- Research Intern at IIT Patna (Feb 2023 to April 2023): Worked on Vision transformer, deep learning, Neural Network
- Machine learning Intern at Ignitus (June 2023 to June 2023): Worked on PCA Algorithm with EDA on NMIST dataset.

Education:
- Bachelor of Technology in Computer Science from AMITY UNIVERSITY (2020–2024)

Certifications:
- Introduction to Generative AI Learning Path - Google_Cloud_Badge
- UHACKATHON 3.0 -UPES Dehradun
- AWS AI ML Deep racer Student league –AWS-deepracer
- AI Programming with Python - Bertelsmann Nanodegree Scholarship on Udacity

Projects:
- Liver segmentation using monai to detect tumor (In-house Internship College): Worked on healthcare imaging with use of AI (Monai and Pytorch).


In [31]:
result = {"score":1, "key_metric": ["balg", "afhsdhjbf"]}
res={"resume_id":"meow", **result}
print(res)

{'resume_id': 'meow', 'score': 1, 'key_metric': ['balg', 'afhsdhjbf']}


In [32]:
match[0]['distance']

[0.9213221669197083, 1.082058310508728]

In [33]:
match[0]['id']

['job_dev_001', 'job_003']

In [34]:
sorted_match=sorted(match, key=lambda k: k['distance'])

In [35]:
sorted_match[0]

{'id': ['job_dev_001', 'job_003'],
 'text': ['We are looking for a Machine Learning Engineer with experience in Python, TensorFlow, and deep learning. The ideal candidate should have worked on computer vision projects and be familiar with neural networks and NLP. Responsibilities include building scalable ML models, collaborating with research teams, and deploying solutions in production.',
  'Join our AI research team as an NLP Engineer. Must have experience with Transformers and PyTorch.'],
 'meta_data': [{'company': 'TechNova AI',
   'role': 'Machine Learning Engineer',
   'location': 'Remote',
   'experience_required': '1+ years',
   'skills_required': 'Python, TensorFlow, Deep Learning, Computer Vision, NLP'},
  {'location': 'Remote',
   'company': 'DeepLang AI',
   'role': 'NLP Engineer',
   'experience_required': '2+ years'}],
 'distance': [0.9213221669197083, 1.082058310508728],
 'collection_name': 'job_post'}

In [36]:
from datetime import datetime
from types import SimpleNamespace

# Simulate a posted_by user
dummy_user = SimpleNamespace(name="Divyansh")

# Create a dummy JobPost-like object
dummy_job = SimpleNamespace(
    title="AI Research Intern",
    description="Join our cutting-edge AI team to work on NLP and computer vision projects.",
    requirements="Python, PyTorch, Transformers, basic ML theory",
    posted_by=dummy_user,
    posted_by_id="user_123",
    status="active",
    created_at=datetime(2025, 11, 6, 18, 30)
)

In [37]:
def preprocess_job_post(job):
        return "\n".join([
            f"Job Title: {job.title}",
            f"Job Description: {job.description}",
            f"Job Requirements: {job.requirements}",
            f"Job Status: {job.status}"
        ])


In [38]:
preprocess_job_post(dummy_job)

'Job Title: AI Research Intern\nJob Description: Join our cutting-edge AI team to work on NLP and computer vision projects.\nJob Requirements: Python, PyTorch, Transformers, basic ML theory\nJob Status: active'

## mock interview

In [39]:
from langchain_community.document_loaders import ( PyPDFLoader, PDFMinerLoader, PyMuPDFLoader, UnstructuredPDFLoader, UnstructuredWordDocumentLoader)
from langchain_core.documents import Document
from typing import List,Dict,Any

import transformers
import re
import os

from google import genai
import pdfplumber
# from transformers.utils.import_utils import candidates
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv('OPENAI_API_KEY')
os.environ['GEMINI_API_KEY']=os.getenv('GEMINI_API_KEY')

In [40]:
job_title='machine learning engineer'
job_description='We are seeking a Machine Learning Engineer to develop scalable ML models for real-time applications. Responsibilities include data preprocessing, model training, deployment, and performance optimization. Proficiency in Python, TensorFlow, and cloud platforms (AWS/GCP) is essential. Experience with NLP or computer vision is a plus.'

job_requirements="We are seeking a Machine Learning Engineer to develop scalable ML models for real-time applications. The role involves data preprocessing, model training, deployment, and performance optimization, along with collaboration across teams to integrate ML solutions into production systems. Candidates must have a Bachelor’s or Master’s degree in Computer Science, Data Science, or a related field, with at least 2 years of experience in machine learning or data science roles. Proficiency in Python and ML frameworks like TensorFlow or PyTorch is essential, as is experience with cloud platforms such as AWS, GCP, or Azure. A strong understanding of data preprocessing, feature engineering, and model evaluation is required, and familiarity with NLP or computer vision techniques is a plus. Excellent problem-solving and communication skills are also expected."
n_easy_questions=2
n_medium_questions=2
n_hard_questions=2

In [41]:
prompt = f"""
    You are an expert AI assistant for interview tasks.
    Your Job is to provide tailored mock interviews for candidates based on the job description and requirements.
    Generate a structured mock interview for the role. The output should be in JSON with keys: easy, medium and hard.
    Respond only in Valid JSON and nothing else.
    Each key maps to a list of questions with the following structure:

    {{
        "question": "",
        "answer": "",
        "difficulty": ""
    }}

    Inputs:
    - Job Title: {job_title}
    - Job Description: {job_description}
    - Job Requirements: {job_requirements}
    - Number of easy questions: {n_easy_questions}
    - Number of medium questions: {n_medium_questions}
    - Number of hard questions: {n_hard_questions}
    - Tone: Professional and Concise

    Rules:
    - Provide exactly the requested number of questions for each category.
    - Each question should be action-result formatted and containes keywords relevant to the job.
    - Keep each question and answer concise (questions <= 30 words).
    - Answer should be a 1-3 sentence summary.
    - Output only in valid JSON.

    """

In [42]:
client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))

response = client.models.generate_content(
    model='gemini-2.0-flash',
    contents=prompt
)

print(response)

# genai.configure(api_key=os.getenv('GEMINI_API_KEY'))
# model = genai.GenerativeModel("gemini-2.0-flash")  # or "gemini-1.5-pro"
#
# response = model.generate_content(prompt)

sdk_http_response=HttpResponse(
  headers=<dict len=11>
) candidates=[Candidate(
  avg_logprobs=-0.29065609349350985,
  content=Content(
    parts=[
      Part(
        text="""```json
{
  "easy": [
    {
      "question": "How have you used Python for data preprocessing in past projects, and what libraries did you leverage?",
      "answer": "I used Python with pandas and scikit-learn to clean and transform data. This involved handling missing values, scaling features, and encoding categorical variables to prepare data for model training.",
      "difficulty": "easy"
    },
    {
      "question": "Describe your experience with model deployment on cloud platforms like AWS or GCP.",
      "answer": "I've deployed models on AWS using SageMaker and Lambda. This included containerizing models, setting up endpoints, and monitoring performance to ensure reliable and scalable deployment.",
      "difficulty": "easy"
    }
  ],
  "medium": [
    {
      "question": "Explain your approach to f

In [43]:
cleaned_json=response.text.strip('`json \n')

In [44]:
cleaned_json=json.loads(cleaned_json)

In [45]:
cleaned_json

{'easy': [{'question': 'How have you used Python for data preprocessing in past projects, and what libraries did you leverage?',
   'answer': 'I used Python with pandas and scikit-learn to clean and transform data. This involved handling missing values, scaling features, and encoding categorical variables to prepare data for model training.',
   'difficulty': 'easy'},
  {'question': 'Describe your experience with model deployment on cloud platforms like AWS or GCP.',
   'answer': "I've deployed models on AWS using SageMaker and Lambda. This included containerizing models, setting up endpoints, and monitoring performance to ensure reliable and scalable deployment.",
   'difficulty': 'easy'}],
 'medium': [{'question': 'Explain your approach to feature engineering for a machine learning model, including specific techniques?',
   'answer': 'I explore domain knowledge, analyze feature importance, and try one-hot encoding. Feature engineering is crucial to model performance and interpretabil

### strength and weakened

In [46]:

self_review="I successfully led the migration of our ML pipeline to AWS, improving model deployment speed by 40%. I also mentored two junior engineers and contributed to the design of our new feature extraction module. However, I struggled with time management during the last sprint and missed a few internal deadlines."

manager_review="The employee demonstrated strong leadership and technical skills during the AWS migration. Their mentorship efforts were valuable to the team. However, time management and sprint planning need improvement. I recommend training in agile methodologies and better workload estimation."



prompt = f"""
        You are an expert performance review summarizer assistant.
        You will be provided with an employee self-assessment and a manager review.
        Your task is to summarize the employee's performance based on the self-assessment and manager review.
        Respond only in valid JSON and nothing else.
        If you cannot answer, return the JSON with empty fields or null.
        Return a JSON object exactly matching the schema below:

        Schema:
        {{
            "Strengths": "",
            "Weaknesses": "",
            "Improvements": "",
            "Actionable_step: "",
            "Comments": ""
        }}

        Inputs:
        - Employee Self Review type: {self_review}
        - Manager Review: {manager_review}

        Rules:
        = Provide a summary of the employee's performance based on the self-assessment and manager review.
        - Provide a concise summary of strengths, weaknesses, and areas for improvement each in less than 20 words.
        - Actionable Steps should be specific and prioritized (e.g. "Take X Courses").
        - Comments should be a single sentence summary.
        - Output only in valid JSON

    """


In [47]:
client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))

response = client.models.generate_content(
    model='gemini-2.0-flash',
    contents=prompt
)
cleaned_json=response.text.strip('`json \n')
cleaned_json=json.loads(cleaned_json)
print(cleaned_json)


{'Strengths': 'Strong leadership, technical skills, and effective mentorship.', 'Weaknesses': 'Time management and meeting internal deadlines.', 'Improvements': 'Needs improvement in time management and sprint planning.', 'Actionable_step': 'Training in agile methodologies and workload estimation.', 'Comments': 'The employee shows promise but needs to improve time management.'}


## tailor resume

In [48]:
target_jobs=['machine learning engineer', 'NLP engineer', 'Data engineer']

prompt = f"""
        You are an expert AI assistant for HR Tasks.
        Respond only in Valid JSON and nothing else.
        If you cannot answer, return the JSON with empty fields or null.

        Rewrite and optimize the candidate's resume bullets to better match the job description.
        The output should be a JSON object with the following structure:

        {{
            "rewritten_summary": "",
            "rewritten_bullets": [],
            "explained_changes": [] # short reasons for the changes
        }}

        Inputs:
        - Resume text: {resume_text}
        - Target Jobs: {target_jobs}
        - Tone: professional and concise

        Rules:
        - Provide a rewritten summary optimized for the target jobs.
        - Provide up to 5-6 rewritten_bullets; each bullet should be action-result formatted and containes keywords relevant to the target job.
        - explained_changes: provide up to 3-4 short explanation for the changes.
        - Output only in valid JSON.

        """

In [49]:
response = client.models.generate_content(
    model='gemini-2.0-flash',
    contents=prompt
)
cleaned_json=response.text.strip('`json \n')
cleaned_json=json.loads(cleaned_json)
print(cleaned_json)


{'rewritten_summary': "Highly motivated and skilled recent graduate with a Bachelor's degree in Computer Science and hands-on experience in machine learning, deep learning, and NLP. Proven ability to develop and implement AI models for various applications, including image segmentation, generative modeling, and text classification. Eager to contribute to innovative projects in machine learning engineering, NLP engineering, or data engineering roles.", 'rewritten_bullets': ['Developed a DCGAN model for realistic human face generation, leveraging deep learning techniques and a dataset of real face images to produce high-quality synthetic images.', 'Engineered a toxic comment classifier using NLP and machine learning, categorizing English comments into six categories (toxic, severe toxic, threat, obscene, insult, and identity hate).', 'Implemented liver segmentation using MONAI and PyTorch for tumor detection in CT/MRI scans, demonstrating proficiency in healthcare imaging and deep learni

## make JD prompt


In [53]:
# pip install langchain-core langchain-openai

from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# 1) System instructions: how to write JDs
SYSTEM = """You are an expert HR copywriter. Write concise, clear, and attractive job descriptions.
Follow this section order and headings exactly:
- Job Title
- Company
- Location
- About Us
- Job Summary
- Key Responsibilities
- Qualifications

Rules:
- Keep paragraphs tight and scannable.
- Use bullet points where appropriate.
- If a field is missing, infer practical defaults without hallucinating facts.
- Keep length between 250–500 words unless instructed otherwise."""

# 2) One few-shot example (from your Robotics Instructor JD)
examples = [
    {
        "input": """Job Title: Robotics Instructor
Company: SE Team 18 Company
Location: Chennai, Tamil Nadu
Notes: After-school programs; kits include LEGO Mindstorms, VEX, Arduino; audience 8–16 years""",
        "output": """### Job Title
Robotics Instructor

### Company
SE Team 18 Company

### Location
Chennai, Tamil Nadu

### About Us
SE Team 18 is a leading provider of STEM education, inspiring future engineers, programmers, and scientists through hands-on learning in robotics.

### Job Summary
We seek a passionate, energetic instructor to lead after-school programs and workshops for students aged 8–16. You’ll deliver our curriculum, guide robot builds, and foster a collaborative, creative classroom.

### Key Responsibilities
- Deliver engaging robotics lessons for ages 8–16.
- Help students assemble, program, and troubleshoot robots using LEGO Mindstorms, VEX, or Arduino.
- Manage classroom dynamics to ensure a safe, positive, productive environment.
- Prepare and maintain materials, equipment, and robotics kits.
- Offer constructive feedback to build problem-solving skills.

### Qualifications
- Strong interest/basic knowledge in robotics, electronics, or programming.
- Experience with children or educational settings preferred.
- Excellent communication and interpersonal skills.
- Ability to explain complex concepts simply.
- Pursuing/completed a degree in Engineering, Computer Science, Education, or related field is a plus."""
    }
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("assistant", "{output}")
])

few_shot = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_prompt
)

# 3) Final prompt template (plugs in your runtime variables)
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    few_shot,
    ("human",
     """Generate a JD with the same structure and tone.

""")
])

# 4) LLM + chain
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.4)
chain = prompt | llm | StrOutputParser()

# 5) Usage example
jd_text = chain.invoke({
    "job_title": "Machine Learning",
    # "company": "Volvoworks",
    # "location": "Bengaluru, Karnataka",
    # "notes": "Focus on batch + streaming; tech: Python, SQL, Airflow, Spark, Kafka; 3–5 yrs exp; fintech domain preferred"
})

print(jd_text)


### Job Title
Robotics Instructor

### Company
SE Team 18 Company

### Location
Chennai, Tamil Nadu

### About Us
At SE Team 18, we are dedicated to cultivating the next generation of innovators through hands-on STEM education. Our programs empower students to explore robotics and technology, fostering creativity and critical thinking.

### Job Summary
We are looking for a dynamic Robotics Instructor to lead our after-school robotics programs for students aged 8–16. In this role, you will inspire young minds by guiding them through exciting projects using LEGO Mindstorms, VEX, and Arduino kits.

### Key Responsibilities
- Facilitate engaging robotics classes for diverse groups of students.
- Instruct students on building, programming, and troubleshooting robots.
- Create a supportive and inclusive learning environment.
- Organize and maintain robotics kits and classroom materials.
- Encourage teamwork and collaboration among students.
- Assess student progress and provide feedback to e